In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files;

In [0]:
import re

path = "/Volumes/workspace/bronze/raw_files/"

def safe_name(col):
    col = col.replace("#", "n ").replace("%", "pct ")
    col = re.sub(r"[^0-9a-zA-Z]+", "_", col.strip().lower())
    return col.strip("_")

for name in ["attendance", "demographics", "income_tract"]:
    df = spark.read.option("header", True).csv(path + f"{name}_raw.csv")
    df = df.toDF(*[safe_name(c) for c in df.columns])     # rename every column
    df.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"workspace.bronze.{name}")
    print("Bronze:", name, "-", len(df.columns), "columns")

Bronze: attendance - 10 columns
Bronze: demographics - 39 columns
Bronze: income_tract - 5 columns


In [0]:
display(dbutils.fs.ls(path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/raw_files/attendance_raw.csv,attendance_raw.csv,9647001,1794340657000
dbfs:/Volumes/workspace/bronze/raw_files/demographics_raw.csv,demographics_raw.csv,1661948,1794340657000
dbfs:/Volumes/workspace/bronze/raw_files/income_raw.csv,income_raw.csv,830896,1794340657000
dbfs:/Volumes/workspace/bronze/raw_files/income_tract_raw.csv,income_tract_raw.csv,147078,1794340931000
dbfs:/Volumes/workspace/bronze/raw_files/locations_raw.json,locations_raw.json,2995584,1794340657000


In [0]:
for name in ["attendance", "demographics", "locations", "income_tract"]:
    (spark.read.parquet(path + f"{name}_clean.parquet")
        .write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"workspace.silver.{name}"))
    print("Silver:", name)

Silver: attendance
Silver: demographics
Silver: locations
Silver: income_tract


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw_files/"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/raw_files/attendance_raw.csv,attendance_raw.csv,9647001,1794340657000
dbfs:/Volumes/workspace/bronze/raw_files/demographics_clean.parquet,demographics_clean.parquet,518550,1794341474000
dbfs:/Volumes/workspace/bronze/raw_files/demographics_raw.csv,demographics_raw.csv,1661948,1794340657000
dbfs:/Volumes/workspace/bronze/raw_files/income_clean.parquet,income_clean.parquet,370917,1794341474000
dbfs:/Volumes/workspace/bronze/raw_files/income_raw.csv,income_raw.csv,830896,1794340657000
dbfs:/Volumes/workspace/bronze/raw_files/income_tract_clean.parquet,income_tract_clean.parquet,25949,1794341474000
dbfs:/Volumes/workspace/bronze/raw_files/income_tract_raw.csv,income_tract_raw.csv,147078,1794340931000
dbfs:/Volumes/workspace/bronze/raw_files/locations_clean.parquet,locations_clean.parquet,92370,1794341474000
dbfs:/Volumes/workspace/bronze/raw_files/locations_raw.json,locations_raw.json,2995584,1794340657000


In [0]:
%sql
DESCRIBE workspace.silver.attendance;

col_name,data_type,comment
school,string,null
schoolyear,string,null
month_code,string,null
calmonth,string,null
gradelevel,string,null
gradesort,string,null
roster_count,string,null
absent,bigint,null
present,bigint,null
released,bigint,null


In [0]:
%sql
SELECT * FROM workspace.silver.attendance LIMIT 5;

school,schoolyear,month_code,calmonth,gradelevel,gradesort,roster_count,absent,present,released,attendance_rate
01M015,2019-20,1,Sep,1,1,26,13,427,0,0.9704545454545455
01M015,2019-20,1,Sep,2,2,38,13,620,0,0.9794628751974723
01M015,2019-20,1,Sep,3,3,35,22,552,0,0.9616724738675958
01M015,2019-20,1,Sep,4,4,30,13,460,0,0.9725158562367865
01M015,2019-20,1,Sep,5,5,21,19,322,0,0.9442815249266863


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.school_year_summary AS
WITH att AS (
    SELECT school      AS dbn,
           schoolyear  AS school_year,
           SUM(present) AS total_present,
           SUM(absent)  AS total_absent,
           SUM(present) / (SUM(present) + SUM(absent)) AS attendance_rate
    FROM workspace.silver.attendance
    GROUP BY school, schoolyear
)
SELECT a.*,
       l.school_name,
       l.borough,
       l.location_category_description AS school_type,
       l.latitude,
       l.longitude,
       d.total_enrollment,
       d.pct_poverty,
       d.economic_need_index,
       i.median_income
FROM att a
LEFT JOIN workspace.silver.locations    l ON a.dbn = l.dbn
LEFT JOIN workspace.silver.demographics d ON a.dbn = d.dbn AND a.school_year = d.year
LEFT JOIN workspace.silver.income_tract i ON l.county = i.county AND l.census_tract = i.census_tract;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*)                         AS school_years,
       COUNT(borough)                   AS matched_location,
       COUNT(pct_poverty)               AS matched_demographics,
       COUNT(median_income)             AS matched_income,
       ROUND(AVG(attendance_rate), 3)   AS avg_attendance
FROM workspace.gold.school_year_summary;

school_years,matched_location,matched_demographics,matched_income,avg_attendance
4745,4740,4187,4227,0.901


In [0]:
%sql
SELECT * FROM workspace.gold.school_year_summary LIMIT 10;

dbn,school_year,total_present,total_absent,attendance_rate,school_name,borough,school_type,latitude,longitude,total_enrollment,pct_poverty,economic_need_index,median_income
01M515,2019-20,42033,16964,0.7124599555909622,Lower East Side Preparatory High School,Manhattan,High school,40.72004,-73.986038,531.0,0.748,null,90222.0
02M011,2019-20,97858,5859,0.9435097428579693,P.S. 011 William T. Harris,Manhattan,Elementary,40.744186,-74.000743,900.0,0.281,0.38,95355.0
02M041,2019-20,73524,3230,0.9579175026708706,P.S. 041 Greenwich Village,Manhattan,Elementary,40.73545,-73.999094,668.0,0.075,0.086,173615.0
02M151,2019-20,50767,3766,0.9309408981717492,Yorkville Community School,Manhattan,Elementary,40.777948,-73.947287,475.0,0.366,0.391,141232.0
02M198,2019-20,53175,3347,0.9407841194579102,P.S. 198 Isador E. Ida Straus,Manhattan,Elementary,40.785018,-73.95042,491.0,0.34600000000000003,0.34600000000000003,159712.0
02M212,2019-20,39398,2803,0.9335797729911613,P.S. 212 Midtown West,Manhattan,Elementary,40.761327,-73.988665,370.0,0.205,0.285,145488.0
02M290,2019-20,62466,3033,0.953693949525947,P.S. 290 Manhattan New School,Manhattan,Elementary,40.775181,-73.952777,573.0,0.124,0.09699999999999999,134167.0
02M439,2019-20,49398,1178,0.9767083201518507,Manhattan Village Academy,Manhattan,High school,40.741963,-73.991966,460.0,0.741,0.659,186191.0
02M586,2019-20,4660,1957,0.704246637448995,Harvey Milk High School,Manhattan,High school,40.729661,-73.992343,61.0,0.77,0.809,236250.0
03M145,2019-20,41069,5293,0.8858332254863898,"P.S. 145, The Bloomingdale School",Manhattan,Elementary,40.799098,-73.965611,395.0,0.706,0.774,37641.0


In [0]:
       ROUND(SUM(present) / (SUM(present) + SUM(absent)), 4) AS attendance_rate
...
       ROUND(d.pct_poverty, 3)          AS pct_poverty,
       ROUND(d.economic_need_index, 3)  AS economic_need_index,

  File <command-8261123584520279>, line 1
    ROUND(SUM(present) / (SUM(present) + SUM(absent)), 4) AS attendance_rate
    ^
IndentationError: unexpected indent
